In [3]:
import os
import sys
import json
import random
from dataclasses import dataclass
from typing import Dict, Any, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.manifold import TSNE
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm
from transformers import AutoTokenizer

PROJECT_ROOT = "/autodl-fs/data/archive"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from models.dataset import build_scaler_streaming, YelpBertDataset
from models.bert_gate_fusion import BERTGateFusionClassifier


class StandardScaler:
    def __init__(self, mean=None, std=None):
        self.mean = np.array(mean, dtype=np.float32) if mean is not None else None
        self.std = np.array(std, dtype=np.float32) if std is not None else None

    def transform(self, x):
        x = np.array(x, dtype=np.float32)
        return (x - self.mean) / (self.std + 1e-8)

    def state_dict(self):
        return {"mean": self.mean.tolist(), "std": self.std.tolist()}

    @classmethod
    def from_state_dict(cls, state):
        return cls(mean=state["mean"], std=state["std"])


def load_scaler_from_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        state = json.load(f)
    return StandardScaler.from_state_dict(state)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_json(obj: Dict[str, Any], path: str):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_jsonl(path: str) -> List[Dict[str, Any]]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


def get_num_classes(task: str) -> int:
    if task == "sentiment":
        return 3
    if task == "rating":
        return 9
    raise ValueError(f"Unsupported task: {task}")


def get_class_names(task: str) -> List[str]:
    if task == "sentiment":
        return ["negative", "neutral", "positive"]
    return [f"rating_{i}" for i in range(get_num_classes(task))]


def load_training_config(output_dir: str) -> Dict[str, Any]:
    cfg_path = os.path.join(output_dir, "config.json")
    if not os.path.exists(cfg_path):
        raise FileNotFoundError(f"未找到训练配置文件: {cfg_path}")
    return load_json(cfg_path)


def try_import_shap():
    try:
        import shap
        return shap
    except Exception as e:
        print(f"[WARN] shap 导入失败，跳过 SHAP: {e}")
        return None


def find_last_linear(model: nn.Module):
    last_name, last_module = None, None
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            last_name, last_module = name, module
    return last_name, last_module


def save_classification_outputs(labels, preds, probs, task, out_dir):
    ensure_dir(out_dir)
    class_names = get_class_names(task)

    report_dict = classification_report(
        labels, preds, target_names=class_names, output_dict=True, zero_division=0
    )
    report_text = classification_report(
        labels, preds, target_names=class_names, digits=4, zero_division=0
    )

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    metrics = {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_precision": float(p_macro),
        "macro_recall": float(r_macro),
        "macro_f1": float(f1_macro),
        "weighted_precision": float(p_weighted),
        "weighted_recall": float(r_weighted),
        "weighted_f1": float(f1_weighted),
    }

    if probs is not None:
        try:
            y_true_bin = label_binarize(labels, classes=list(range(len(class_names))))
            if y_true_bin.shape[1] == 1:
                y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
            metrics["macro_ovr_auc"] = float(
                roc_auc_score(y_true_bin, probs, average="macro", multi_class="ovr")
            )
        except Exception:
            metrics["macro_ovr_auc"] = None

    save_json(metrics, os.path.join(out_dir, "metrics.json"))
    save_json(report_dict, os.path.join(out_dir, "classification_report.json"))
    with open(os.path.join(out_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(report_text)

    cm = confusion_matrix(labels, preds)
    cm_norm = confusion_matrix(labels, preds, normalize="true")
    np.save(os.path.join(out_dir, "confusion_matrix.npy"), cm)
    np.save(os.path.join(out_dir, "confusion_matrix_normalized.npy"), cm_norm)

    def plot_cm(matrix, title, save_path, is_float=False):
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111)
        im = ax.imshow(matrix, aspect="auto")
        plt.colorbar(im)
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=45, ha="right")
        ax.set_yticklabels(class_names)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(title)

        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                val = matrix[i, j]
                txt = f"{val:.2f}" if is_float else f"{int(val)}"
                ax.text(j, i, txt, ha="center", va="center")

        plt.tight_layout()
        fig.savefig(save_path, dpi=220, bbox_inches="tight")
        plt.close(fig)

    plot_cm(cm, "Confusion Matrix", os.path.join(out_dir, "confusion_matrix.png"), False)
    plot_cm(
        cm_norm,
        "Normalized Confusion Matrix",
        os.path.join(out_dir, "confusion_matrix_normalized.png"),
        True,
    )
    return metrics, report_text


def save_prediction_table(raw_records, labels, preds, probs, out_dir):
    rows = []
    for i, (y_true, y_pred) in enumerate(zip(labels, preds)):
        row = {
            "index": i,
            "true_label": int(y_true),
            "pred_label": int(y_pred),
            "correct": int(y_true == y_pred),
        }

        if i < len(raw_records):
            rec = raw_records[i]
            row["text"] = rec.get("text", "")
            if "meta_features" in rec:
                row["meta_features"] = json.dumps(rec.get("meta_features", []), ensure_ascii=False)

        if probs is not None:
            for c in range(probs.shape[1]):
                row[f"prob_{c}"] = float(probs[i, c])

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out_dir, "predictions.csv"), index=False, encoding="utf-8-sig")
    df[df["correct"] == 0].to_csv(
        os.path.join(out_dir, "misclassified_cases.csv"),
        index=False,
        encoding="utf-8-sig",
    )

    error_df = df[df["correct"] == 0].copy()
    if len(error_df) > 0:
        summary = (
            error_df.groupby(["true_label", "pred_label"])
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        summary.to_csv(
            os.path.join(out_dir, "error_pair_summary.csv"),
            index=False,
            encoding="utf-8-sig",
        )


def run_tsne_and_plot(features: np.ndarray, labels: List[int], task: str, out_path: str, title: str):
    if features is None or len(features) < 3:
        print("[WARN] 样本太少，跳过 t-SNE")
        return

    perplexity = max(2, min(30, len(features) // 4))
    tsne = TSNE(
        n_components=2,
        random_state=42,
        init="pca",
        learning_rate="auto",
        perplexity=perplexity,
    )
    emb = tsne.fit_transform(features)
    class_names = get_class_names(task)

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111)
    labels_np = np.array(labels)

    for c in sorted(np.unique(labels_np)):
        idx = labels_np == c
        ax.scatter(emb[idx, 0], emb[idx, 1], s=18, alpha=0.75, label=class_names[int(c)])

    ax.set_title(title)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(loc="best", fontsize=9)
    plt.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    np.save(out_path.replace(".png", ".npy"), emb)


def bert_collate_with_meta(batch, pad_token_id=0):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels, meta_features = [], [], [], []

    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(
            torch.cat([x["input_ids"], torch.full((pad_len,), pad_token_id, dtype=torch.long)])
        )
        attention_mask.append(
            torch.cat([x["attention_mask"], torch.zeros(pad_len, dtype=torch.long)])
        )
        labels.append(x["label"])
        meta_features.append(x["meta_features"])

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels),
        "meta_features": torch.stack(meta_features),
    }


@dataclass
class EvalArgs:
    task: str = "sentiment"
    output_dir: str = "outputs/checkpoints/bert_gate_fusion"
    eval_dir: str = "eval/bert_gate_fusion"
    batch_size: int = 16
    num_workers: int = 0
    pin_memory: bool = False
    seed: int = 42
    shap_max_eval_samples: int = 64
    shap_background_size: int = 20


args = EvalArgs()


def build_model_from_cfg(cfg, device, output_dir):
    tokenizer = AutoTokenizer.from_pretrained(cfg["pretrained_model_name"])

    scaler_path = os.path.join(output_dir, "scaler.json")
    if os.path.exists(scaler_path):
        print(f"[INFO] Load scaler from: {scaler_path}")
        scaler = load_scaler_from_json(scaler_path)
    else:
        print("[WARN] scaler.json 不存在，回退为重新构建 scaler")
        scaler = build_scaler_streaming(cfg["train_path"])

    test_dataset = YelpBertDataset(
        cfg["test_path"],
        cfg["task"],
        tokenizer,
        cfg["max_length"],
        cfg["text_field"],
        True,
        scaler,
    )
    sample_item = test_dataset[0]
    meta_dim = sample_item["meta_features"].shape[0]

    model = BERTGateFusionClassifier(
        pretrained_model_name=cfg["pretrained_model_name"],
        num_classes=get_num_classes(cfg["task"]),
        meta_dim=meta_dim,
        fusion_dim=cfg["fusion_dim"],
        dropout=cfg["dropout"],
        freeze_bert=cfg.get("freeze_bert", False),
    ).to(device)

    return model, tokenizer, scaler


def make_dataloader(cfg, tokenizer, scaler, batch_size, num_workers, pin_memory):
    test_dataset = YelpBertDataset(
        cfg["test_path"],
        cfg["task"],
        tokenizer,
        cfg["max_length"],
        cfg["text_field"],
        True,
        scaler,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=lambda b: bert_collate_with_meta(b, tokenizer.pad_token_id),
    )
    return test_dataset, test_loader


def evaluate(model, dataloader, criterion, device, tsne_limit=1500):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    all_features = []
    cached_batches = []

    last_name, last_linear = find_last_linear(model)
    hook_cache = {"features": None}
    hook_handle = None

    if last_linear is not None:
        def hook_fn(module, inputs, outputs):
            x = inputs[0]
            if isinstance(x, tuple):
                x = x[0]
            hook_cache["features"] = x.detach().cpu()

        hook_handle = last_linear.register_forward_hook(hook_fn)
        print(f"[INFO] t-SNE hook module: {last_name}")

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Evaluating", dynamic_ncols=True)
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            meta_features = batch["meta_features"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                meta_features=meta_features,
            )
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(logits, dim=-1)

            total_loss += loss.item() * labels.size(0)
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_probs.append(probs.cpu().numpy())

            if hook_cache["features"] is not None and len(all_features) < tsne_limit:
                feat = hook_cache["features"]
                remain = tsne_limit - len(all_features)
                feat = feat[:remain]
                all_features.extend(feat.numpy())

            if len(cached_batches) < 1:
                cached_batches.append(
                    {
                        "input_ids": input_ids.cpu(),
                        "attention_mask": attention_mask.cpu(),
                        "meta_features": meta_features.cpu(),
                    }
                )

            running_loss = total_loss / max(1, len(all_labels))
            running_acc = accuracy_score(all_labels, all_preds)
            pbar.set_postfix(loss=f"{running_loss:.4f}", acc=f"{running_acc:.4f}")

    if hook_handle is not None:
        hook_handle.remove()

    return {
        "loss": total_loss / max(1, len(all_labels)),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "labels": all_labels,
        "predictions": all_preds,
        "probabilities": np.concatenate(all_probs, axis=0) if all_probs else None,
        "features": np.array(all_features) if all_features else None,
        "cached_batches": cached_batches,
    }


def run_meta_shap(model, cached_batches, out_dir, max_eval_samples=32, background_size=8):
    shap = try_import_shap()
    if shap is None or not cached_batches:
        return

    # 只取第一个 batch，避免不同 batch 动态 padding 长度不一致
    batch0 = cached_batches[0]

    device = next(model.parameters()).device
    input_ids = batch0["input_ids"]
    attention_mask = batch0["attention_mask"]
    meta_features = batch0["meta_features"]

    total_n = meta_features.size(0)
    if total_n < 2:
        print("[WARN] SHAP 样本太少，跳过")
        return

    n = min(max_eval_samples, total_n)
    bg_n = min(background_size, max(2, n // 2), total_n)

    input_ids = input_ids[:n].to(device)
    attention_mask = attention_mask[:n].to(device)
    meta_eval = meta_features[:n].cpu().numpy().astype(np.float32)
    meta_bg = meta_features[:bg_n].cpu().numpy().astype(np.float32)

    def predict_meta(meta_np):
        model.eval()
        meta_np = np.array(meta_np, dtype=np.float32)
        cur = torch.tensor(meta_np, dtype=torch.float32, device=device)

        cur_bs = cur.size(0)

        # 固定文本输入，只截取相同 batch 大小
        ids = input_ids[:cur_bs]
        masks = attention_mask[:cur_bs]

        with torch.no_grad():
            logits = model(
                input_ids=ids,
                attention_mask=masks,
                meta_features=cur,
            )
            probs = torch.softmax(logits, dim=-1).cpu().numpy()

        return probs

    try:
        explainer = shap.KernelExplainer(predict_meta, meta_bg)
        shap_values = explainer.shap_values(
            meta_eval,
            nsamples=min(50, 2 * meta_eval.shape[1] + 10),
        )

        class_id = 0
        if isinstance(shap_values, list):
            mean_abs = [np.abs(v).mean() for v in shap_values]
            class_id = int(np.argmax(mean_abs))
            values = shap_values[class_id]
        else:
            if getattr(shap_values, "ndim", 2) == 3:
                mean_abs = np.abs(shap_values).mean(axis=(0, 1))
                class_id = int(np.argmax(mean_abs))
                values = shap_values[:, :, class_id]
            else:
                values = shap_values

        feature_names = [f"meta_{i}" for i in range(meta_eval.shape[1])]

        plt.figure(figsize=(10, 6))
        shap.summary_plot(values, meta_eval, feature_names=feature_names, show=False)
        plt.title(f"SHAP Summary (class={class_id})")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "shap_meta_summary.png"), dpi=220, bbox_inches="tight")
        plt.close()

        np.save(os.path.join(out_dir, "shap_meta_values.npy"), np.array(values))
        save_json({"shap_target_class": class_id}, os.path.join(out_dir, "shap_meta_info.json"))
        print(f"[INFO] SHAP 已保存到: {out_dir}")

    except Exception as e:
        print(f"[WARN] SHAP 运行失败，已跳过: {e}")


def main(eval_args: EvalArgs):
    set_seed(eval_args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Device: {device}")

    cfg = load_training_config(eval_args.output_dir)
    model, tokenizer, scaler = build_model_from_cfg(cfg, device, eval_args.output_dir)
    _, test_loader = make_dataloader(
        cfg, tokenizer, scaler, eval_args.batch_size, eval_args.num_workers, eval_args.pin_memory
    )

    best_model_path = os.path.join(eval_args.output_dir, "best_model.pt")
    if not os.path.exists(best_model_path):
        raise FileNotFoundError(f"未找到模型权重: {best_model_path}")

    state = torch.load(best_model_path, map_location=device)
    model.load_state_dict(state)

    criterion = nn.CrossEntropyLoss()
    results = evaluate(model, test_loader, criterion, device)

    out_dir = eval_args.eval_dir
    ensure_dir(out_dir)
    raw_records = load_jsonl(cfg["test_path"])

    summary, report_text = save_classification_outputs(
        results["labels"],
        results["predictions"],
        results["probabilities"],
        cfg["task"],
        out_dir,
    )

    save_prediction_table(
        raw_records,
        results["labels"],
        results["predictions"],
        results["probabilities"],
        out_dir,
    )

    if results["features"] is not None and len(results["features"]) > 10:
        run_tsne_and_plot(
            results["features"],
            results["labels"][:len(results["features"])],
            cfg["task"],
            os.path.join(out_dir, "tsne_penultimate_features.png"),
            "t-SNE of Penultimate Features",
        )

    run_meta_shap(
        model,
        results["cached_batches"],
        out_dir,
        eval_args.shap_max_eval_samples,
        eval_args.shap_background_size,
    )

    final_payload = {
        "test_loss": float(results["loss"]),
        "test_accuracy": float(results["accuracy"]),
        "test_macro_f1": float(results["macro_f1"]),
        "extra_metrics": summary,
    }
    save_json(final_payload, os.path.join(out_dir, "eval_summary.json"))

    print("\n========== Final Eval Results ==========")
    print(json.dumps(final_payload, ensure_ascii=False, indent=2))
    print("\n========== Classification Report ==========")
    print(report_text)
    print(f"\n[INFO] 评估结果已保存到: {out_dir}")


if __name__ == "__main__":
    main(args)

[INFO] Device: cuda
[INFO] Load scaler from: outputs/checkpoints/bert_gate_fusion/scaler.json


Loading test.jsonl: 0it [00:00, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading test.jsonl: 0it [00:00, ?it/s]

[INFO] t-SNE hook module: classifier


Evaluating:   0%|          | 0/12500 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

[WARN] SHAP 运行失败，已跳过: Sizes of tensors must match except in dimension 1. Expected size 16 but got size 384 for tensor number 1 in the list.

========== Final Eval Results ==========
{
  "test_loss": 0.26722617042725205,
  "test_accuracy": 0.91364,
  "test_macro_f1": 0.8110417451583686,
  "extra_metrics": {
    "accuracy": 0.91364,
    "macro_precision": 0.8266363841390235,
    "macro_recall": 0.8001362595387231,
    "macro_f1": 0.8110417451583686,
    "weighted_precision": 0.9078268385421537,
    "weighted_recall": 0.91364,
    "weighted_f1": 0.9099640203326239,
    "macro_ovr_auc": 0.9712221280615796
  }
}

========== Classification Report ==========
              precision    recall  f1-score   support

    negative     0.8939    0.9208    0.9071     46266
     neutral     0.6327    0.5086    0.5639     19800
    positive     0.9533    0.9711    0.9621    133934

    accuracy                         0.9136    200000
   macro avg     0.8266    0.8001    0.8110    200000
weighted avg  